# Notebook 07: Audio - Automatic Speech Recognition (ASR)

## Learning Objectives

In this notebook, you will learn:
- How to convert speech to text using ASR models
- How OpenAI's Whisper model works for transcription
- How to handle different audio formats and sources
- How to get word-level and segment-level timestamps
- How to transcribe in 99+ languages
- How to evaluate transcription accuracy with Word Error Rate (WER)

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **Tiny (CPU)** | openai/whisper-tiny | 72MB | 4GB | Any CPU | Fastest, lower accuracy |
| **Small (CPU)** | openai/whisper-small | 461MB | 8GB | 8GB RAM, CPU | Good balance |
| **Medium (GPU)** | openai/whisper-medium | 1.5GB | 8GB | 8GB+ VRAM | Production quality |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `soundfile`, `librosa`
- Optional: `datasets` for evaluation on LibriSpeech

## Expected Behaviors

**First Time Running**:
- Whisper-tiny download: ~72MB (smallest model in this tutorial)
- Very fast even on CPU (~32x realtime)
- Cached after first download

**Transcription Output**:
```python
{'text': 'The quick brown fox jumped over the lazy dog.'}
```

**With Timestamps**:
```python
{'text': '...', 'chunks': [{'timestamp': (0.0, 2.5), 'text': 'The quick brown fox'}]}
```

**Performance**:
- Whisper-tiny on CPU: ~32x realtime (1 min audio in ~2s)
- Whisper-small on CPU: ~6x realtime (1 min audio in ~10s)
- Whisper-medium on GPU: ~50x realtime

## Overview

**Automatic Speech Recognition (ASR)** converts spoken language into text.

**Use Cases**: Transcription services, voice assistants, meeting notes, accessibility, subtitle generation, voice search.

### Whisper Architecture

Whisper is an encoder-decoder transformer trained on 680,000 hours of multilingual audio:

1. **Audio Preprocessing**: Raw audio -> Log-Mel spectrogram (80 frequency bins, 30s chunks)
2. **Encoder**: Processes the spectrogram into contextual audio embeddings
3. **Decoder**: Auto-regressively generates text tokens, conditioned on encoder output

Key features:
- **Multilingual**: 99+ languages with automatic language detection
- **Robust**: Trained on diverse internet audio (noisy, accented, varied)
- **Timestamps**: Can provide segment-level or word-level timing
- **Translation**: Can translate non-English audio directly to English text

### Whisper Model Family

| Model | Parameters | Size | Relative Speed | English WER |
|-------|-----------|------|----------------|-------------|
| tiny | 39M | 72MB | 32x | ~7.6% |
| base | 74M | 139MB | 16x | ~5.7% |
| small | 244M | 461MB | 6x | ~4.4% |
| medium | 769M | 1.5GB | 2x | ~3.6% |
| large-v3 | 1.55B | 3.1GB | 1x | ~2.9% |

## Setup and Installation

In [ ]:
import sys, time, random, os, warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import pipeline, set_seed

warnings.filterwarnings("ignore")

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")

## Model Selection

In [ ]:
# Option 1: Tiny model (CPU-friendly, fastest, 99+ languages)
MODEL_NAME = "openai/whisper-tiny"  # 72MB, ~32x realtime on CPU

# Option 2: Small model (better accuracy, still CPU-friendly)
# MODEL_NAME = "openai/whisper-small"  # 461MB, ~6x realtime on CPU

# Option 3: Medium model (GPU-optimized, production quality)
# MODEL_NAME = "openai/whisper-medium"  # 1.5GB

print(f"Selected model: {MODEL_NAME}")

## Method 1: Pipeline API (Simplest)

The Pipeline API provides the easiest way to transcribe audio.

In [ ]:
try:
    print(f"Loading {MODEL_NAME}...")
    asr = pipeline(
        "automatic-speech-recognition",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
    )
    print("ASR pipeline loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Ensure you have internet access and sufficient memory.")

### Basic Transcription

We can transcribe directly from a URL or a local file path.

In [ ]:
SAMPLE_AUDIO_URL = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"

start = time.perf_counter()
result = asr(SAMPLE_AUDIO_URL)
elapsed = time.perf_counter() - start

print("Basic Transcription")
print("=" * 70)
print(f"Text: {result['text']}")
print(f"Time: {elapsed:.2f}s")

### Transcription with Timestamps

Whisper can provide segment-level timestamps, useful for subtitle generation and audio navigation.

In [ ]:
result_ts = asr(SAMPLE_AUDIO_URL, return_timestamps=True)

print(f"Full text: {result_ts['text']}\n")

if "chunks" in result_ts:
    chunk_df = pd.DataFrame([
        {
            "Start (s)": f"{chunk['timestamp'][0]:.1f}" if chunk['timestamp'][0] is not None else "N/A",
            "End (s)": f"{chunk['timestamp'][1]:.1f}" if chunk['timestamp'][1] is not None else "N/A",
            "Text": chunk["text"].strip(),
        }
        for chunk in result_ts["chunks"]
    ])
    print("Segments with Timestamps")
    display(chunk_df)
else:
    print("No timestamp chunks returned (may depend on model version).")

### Transcribing Local Audio Files

If you have audio files in the `sample_data/` directory, this cell will transcribe them.

In [ ]:
def transcribe_local_files(directory: str) -> pd.DataFrame:
    """Transcribe all audio files in a directory.

    Args:
        directory: Path to directory containing audio files.

    Returns:
        DataFrame with filename, transcription, and processing time.
    """
    audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
    rows = []

    if not os.path.exists(directory):
        print(f"Directory not found: {directory}")
        return pd.DataFrame()

    audio_files = [
        f for f in os.listdir(directory)
        if os.path.splitext(f)[1].lower() in audio_extensions
    ]

    if not audio_files:
        print(f"No audio files found in {directory}")
        return pd.DataFrame()

    for filename in audio_files:
        filepath = os.path.join(directory, filename)
        try:
            start = time.perf_counter()
            result = asr(filepath)
            elapsed = time.perf_counter() - start
            rows.append({
                "File": filename,
                "Transcription": result["text"][:80],
                "Time (s)": f"{elapsed:.2f}",
            })
        except Exception as e:
            rows.append({"File": filename, "Transcription": f"Error: {e}", "Time (s)": "N/A"})

    return pd.DataFrame(rows)


sample_dir = os.path.join("..", "sample_data")
local_df = transcribe_local_files(sample_dir)
if not local_df.empty:
    print("Local Audio Transcriptions")
    display(local_df)

## Method 2: Manual Model Loading

For more control over generation parameters, we can load the model and processor separately.

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import soundfile as sf
import requests
from io import BytesIO


def load_audio_from_url(url: str) -> tuple[np.ndarray, int]:
    """Download and load audio from a URL.

    Args:
        url: URL pointing to an audio file.

    Returns:
        Tuple of (audio_array, sample_rate).
    """
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    audio, sample_rate = sf.read(BytesIO(response.content))
    return audio, sample_rate


def transcribe_manual(
    audio: np.ndarray,
    sample_rate: int,
    language: str | None = None,
    task: str = "transcribe",
) -> str:
    """Transcribe audio using manual model loading.

    Args:
        audio: Audio waveform as numpy array.
        sample_rate: Sample rate of the audio.
        language: Language code (e.g., 'en', 'fr'). None for auto-detect.
        task: 'transcribe' or 'translate' (translate to English).

    Returns:
        Transcribed text string.
    """
    asr_processor = AutoProcessor.from_pretrained(MODEL_NAME)
    asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_NAME).to(device)

    inputs = asr_processor(
        audio, sampling_rate=sample_rate, return_tensors="pt"
    ).input_features.to(device)

    forced_decoder_ids = None
    if language:
        forced_decoder_ids = asr_processor.get_decoder_prompt_ids(
            language=language, task=task
        )

    with torch.no_grad():
        predicted_ids = asr_model.generate(
            inputs, forced_decoder_ids=forced_decoder_ids
        )

    return asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]


# Test manual transcription
audio, sr = load_audio_from_url(SAMPLE_AUDIO_URL)
print(f"Audio: {len(audio)/sr:.1f}s, {sr}Hz")

text = transcribe_manual(audio, sr, language="en")
print(f"Transcription: {text}")

## Practical Applications

### Application 1: Evaluate on LibriSpeech Dataset

LibriSpeech is a standard ASR benchmark. We use the `clean` test split for evaluation.

In [ ]:
def evaluate_on_librispeech(num_samples: int = 10) -> pd.DataFrame:
    """Evaluate Whisper on LibriSpeech ASR dataset.

    Args:
        num_samples: Number of samples to evaluate.

    Returns:
        DataFrame with reference text, predicted text, and WER.
    """
    try:
        from datasets import load_dataset

        print("Loading LibriSpeech ASR dummy dataset...")
        ds = load_dataset(
            "hf-internal-testing/librispeech_asr_dummy",
            "clean",
            split="validation",
        )
        print(f"Dataset loaded: {len(ds)} samples")

        rows = []
        total_words = 0
        total_errors = 0

        for i in range(min(num_samples, len(ds))):
            sample = ds[i]
            audio = sample["audio"]["array"]
            sr = sample["audio"]["sampling_rate"]
            reference = sample["text"].lower()

            result = asr({"raw": audio, "sampling_rate": sr})
            predicted = result["text"].strip().lower()

            # Simple word error rate
            ref_words = reference.split()
            pred_words = predicted.split()
            errors = abs(len(ref_words) - len(pred_words))
            errors += sum(1 for r, p in zip(ref_words, pred_words) if r != p)
            wer = errors / max(len(ref_words), 1)
            total_words += len(ref_words)
            total_errors += errors

            rows.append({
                "#": i + 1,
                "Reference": reference[:50],
                "Predicted": predicted[:50],
                "WER": f"{wer:.2%}",
            })

        df = pd.DataFrame(rows)
        overall_wer = total_errors / max(total_words, 1)
        print(f"\nOverall WER: {overall_wer:.2%} ({total_errors} errors / {total_words} words)")
        return df

    except Exception as e:
        print(f"Could not load LibriSpeech: {e}")
        return pd.DataFrame()


print("LibriSpeech Evaluation")
evaluate_on_librispeech(10)

### Application 2: Language Detection and Translation

Whisper can automatically detect the spoken language and optionally translate to English.

In [ ]:
language_df = pd.DataFrame([
    {"Language": "English", "Code": "en", "Support Level": "Excellent"},
    {"Language": "Spanish", "Code": "es", "Support Level": "Excellent"},
    {"Language": "French", "Code": "fr", "Support Level": "Excellent"},
    {"Language": "German", "Code": "de", "Support Level": "Excellent"},
    {"Language": "Chinese", "Code": "zh", "Support Level": "Good"},
    {"Language": "Japanese", "Code": "ja", "Support Level": "Good"},
    {"Language": "Korean", "Code": "ko", "Support Level": "Good"},
    {"Language": "Arabic", "Code": "ar", "Support Level": "Good"},
    {"Language": "Hindi", "Code": "hi", "Support Level": "Moderate"},
])

print("Whisper Language Support (sample of 99+ languages)")
language_df

### Application 3: Batch Audio Processing

In [ ]:
def batch_transcribe(audio_sources: list[str]) -> pd.DataFrame:
    """Transcribe multiple audio sources.

    Args:
        audio_sources: List of URLs or file paths to audio files.

    Returns:
        DataFrame with source, transcription, and timing.
    """
    rows = []
    for i, source in enumerate(audio_sources, 1):
        try:
            start = time.perf_counter()
            result = asr(source)
            elapsed = time.perf_counter() - start
            rows.append({
                "#": i,
                "Source": source.split("/")[-1][:30],
                "Text": result["text"][:60],
                "Time (s)": f"{elapsed:.2f}",
            })
        except Exception as e:
            rows.append({"#": i, "Source": source.split("/")[-1][:30], "Text": f"Error: {e}", "Time (s)": "N/A"})

    return pd.DataFrame(rows)


# Test with the same URL (in practice, you'd have different audio files)
print("Batch Transcription")
batch_transcribe([SAMPLE_AUDIO_URL, SAMPLE_AUDIO_URL])

## Performance Benchmarking

Let's measure transcription speed and compare model sizes.

In [ ]:
def benchmark_asr(n_runs: int = 5) -> pd.DataFrame:
    """Benchmark ASR inference speed.

    Args:
        n_runs: Number of runs to average over.

    Returns:
        DataFrame with timing statistics.
    """
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        asr(SAMPLE_AUDIO_URL)
        times.append(time.perf_counter() - start)

    return pd.DataFrame([
        {"Metric": "Model", "Value": MODEL_NAME.split('/')[-1]},
        {"Metric": "Device", "Value": str(device)},
        {"Metric": "Mean (ms)", "Value": f"{np.mean(times) * 1000:.1f}"},
        {"Metric": "Std (ms)", "Value": f"{np.std(times) * 1000:.1f}"},
        {"Metric": "Min (ms)", "Value": f"{np.min(times) * 1000:.1f}"},
        {"Metric": "Max (ms)", "Value": f"{np.max(times) * 1000:.1f}"},
    ])


print("ASR Inference Benchmark")
benchmark_asr()

In [ ]:
# Whisper model family comparison
model_comparison_df = pd.DataFrame([
    {"Model": "whisper-tiny", "Parameters": "39M", "Size": "72MB", "Relative Speed": "32x", "English WER": "~7.6%", "Best For": "Quick demos, edge devices"},
    {"Model": "whisper-base", "Parameters": "74M", "Size": "139MB", "Relative Speed": "16x", "English WER": "~5.7%", "Best For": "Balanced speed/accuracy"},
    {"Model": "whisper-small", "Parameters": "244M", "Size": "461MB", "Relative Speed": "6x", "English WER": "~4.4%", "Best For": "Good accuracy on CPU"},
    {"Model": "whisper-medium", "Parameters": "769M", "Size": "1.5GB", "Relative Speed": "2x", "English WER": "~3.6%", "Best For": "Production with GPU"},
    {"Model": "whisper-large-v3", "Parameters": "1.55B", "Size": "3.1GB", "Relative Speed": "1x", "English WER": "~2.9%", "Best For": "Maximum accuracy"},
])

print("Whisper Model Family")
model_comparison_df

## Error Analysis

Let's explore common ASR failure modes using synthetic and edge-case audio.

In [ ]:
def analyze_failure_modes() -> pd.DataFrame:
    """Test ASR on challenging audio conditions.

    Returns:
        DataFrame with test case descriptions and expected behaviors.
    """
    test_cases = []

    # Test 1: Silence
    silence = np.zeros(16000, dtype=np.float32)  # 1 second of silence
    result = asr({"raw": silence, "sampling_rate": 16000})
    test_cases.append({
        "Test": "1s silence",
        "Expected": "Empty or blank",
        "Output": result["text"].strip()[:60] if result["text"].strip() else "(empty)",
    })

    # Test 2: White noise
    noise = np.random.randn(16000).astype(np.float32) * 0.1
    result = asr({"raw": noise, "sampling_rate": 16000})
    test_cases.append({
        "Test": "White noise",
        "Expected": "Empty or hallucinated",
        "Output": result["text"].strip()[:60] if result["text"].strip() else "(empty)",
    })

    # Test 3: Very short audio (0.1s)
    short = np.zeros(1600, dtype=np.float32)  # 0.1 seconds
    result = asr({"raw": short, "sampling_rate": 16000})
    test_cases.append({
        "Test": "0.1s audio",
        "Expected": "Empty or error",
        "Output": result["text"].strip()[:60] if result["text"].strip() else "(empty)",
    })

    # Test 4: Pure tone (440Hz sine wave)
    t = np.linspace(0, 1, 16000, dtype=np.float32)
    tone = 0.5 * np.sin(2 * np.pi * 440 * t)
    result = asr({"raw": tone, "sampling_rate": 16000})
    test_cases.append({
        "Test": "440Hz sine tone",
        "Expected": "Empty or hallucinated",
        "Output": result["text"].strip()[:60] if result["text"].strip() else "(empty)",
    })

    return pd.DataFrame(test_cases)


print("Error Analysis: Challenging Audio Inputs")
print("Whisper may hallucinate text on non-speech audio.\n")
analyze_failure_modes()

**Observations**: Whisper is known to hallucinate on silence and noise -- it may generate plausible-sounding text even when no speech is present. This is a known limitation that matters for production systems. Solutions include voice activity detection (VAD) preprocessing to filter out non-speech segments before transcription.

## State-of-the-Art ASR Models

In [ ]:
sota_df = pd.DataFrame([
    {"Model": "Whisper (OpenAI)", "Size": "72MB-3.1GB", "Languages": "99+", "WER": "~2.9-7.6%", "Best For": "General multilingual ASR"},
    {"Model": "Wav2Vec 2.0 (Meta)", "Size": "~1.2GB", "Languages": "50+", "WER": "~3.4%", "Best For": "Low-resource languages"},
    {"Model": "Conformer (Google)", "Size": "~600MB", "Languages": "English", "WER": "~2.1%", "Best For": "Streaming ASR"},
    {"Model": "Canary (NVIDIA)", "Size": "~1GB", "Languages": "Multi", "WER": "~3.0%", "Best For": "Enterprise, fast inference"},
])

print("ASR Model Landscape")
sota_df

## Exercises

1. **Custom Audio**: Record your own voice and transcribe it. Compare accuracy for clear vs noisy recordings.

2. **Language Testing**: Test Whisper with non-English audio. Try the `language` parameter for forced language detection.

3. **Model Comparison**: Compare whisper-tiny vs whisper-small on the same audio. Measure speed and accuracy tradeoffs.

4. **Translation**: Use `task="translate"` to translate non-English audio directly to English text.

5. **Long Audio**: Test with audio longer than 30 seconds. Whisper processes in 30s chunks -- observe how it handles boundaries.

## Key Takeaways

- **Whisper** is a powerful multilingual ASR model supporting 99+ languages
- Model sizes range from 72MB (tiny) to 3.1GB (large-v3) with corresponding accuracy tradeoffs
- Whisper can provide **segment-level timestamps** for subtitle generation
- It can also **translate** non-English speech directly to English text
- Known limitation: **hallucination** on silence and non-speech audio
- For production: use Voice Activity Detection (VAD) to filter non-speech before transcription

## Next Steps & Resources

**Next Notebook**: Notebook 08 -- Text-to-Speech with SpeechT5

**Documentation**:
- [Whisper Paper](https://arxiv.org/abs/2212.04356)
- [HuggingFace Whisper Guide](https://huggingface.co/docs/transformers/model_doc/whisper)
- [OpenAI Whisper GitHub](https://github.com/openai/whisper)
- [LibriSpeech Dataset](https://www.openslr.org/12/)